#request resources + use analy environment


salloc --partition test --time 0-04:00 --mem 20gb
module load python
mamba activate analy
cd /n/home07/than157/desktop/done-large_projects/learn-better/evolm/finetune/llama-factory/
jupyter notebook --no-browser --ip=0.0.0.0 --port=8888

In [1]:
from datasets import load_dataset
import pandas as pd
import json
from tqdm import tqdm

### load data

##### dataset 1

In [2]:
#load training set
dbl_train = load_dataset(path="dongboklee/MMLU-Pro-Llama-3.1-70B-Instruct-CoT", split="test")
dbl_train_df = dbl_train.to_pandas()
print(dbl_train_df.shape)

print("# of unique questions:", dbl_train_df['question'].nunique())


(130811, 9)
# of unique questions: 10532


In [3]:
#load test set
dbl_test = load_dataset(path="dongboklee/MMLU-Pro-Llama-3.1-70B-Instruct-CoT", split="validation")
dbl_test_df = dbl_test.to_pandas()
print(dbl_test_df.shape)

print("# of unique questions:", dbl_test_df['question'].nunique())


(840, 9)
# of unique questions: 67


#### dataset 2

In [4]:
#training set
uw_train = load_dataset(path="UW-Madison-Lee-Lab/MMLU-Pro-CoT-Train-Labeled", split="train")
uw_train_df = uw_train.to_pandas()
print(uw_train_df.shape)

# #samples with correct answer
n_rows_with_correct_answer = uw_train_df[uw_train_df['parsed_answer_correctness']].shape[0]
print(f"# of rows with correct answer: {n_rows_with_correct_answer}")

#extract question only
uw_train_df['raw_question'] = uw_train_df['question'].str.split('Question: ').str[1].str.split('\nA. ').str[0]
print("# of unique questions:", uw_train_df['question'].nunique())
print("# of unique questions, raw questions:", uw_train_df['raw_question'].nunique())

(84098, 10)
# of rows with correct answer: 43900
# of unique questions: 5750
# of unique questions, raw questions: 5669


In [5]:
uw_test = load_dataset(path="UW-Madison-Lee-Lab/MMLU-Pro-CoT-Eval", split="test")
uw_test_df = uw_test.to_pandas()

print(uw_test_df.shape)
# #samples with correct answer
n_rows_with_correct_answer = uw_test_df[uw_test_df['parsed_answer_correctness']].shape[0]
print(f"# of rows with correct answer: {n_rows_with_correct_answer}")

#extract question only
uw_test_df['raw_question'] = uw_test_df['question'].str.split('Question: ').str[1].str.split('\nA. ').str[0]
print("# of unique questions:", uw_test_df['question'].nunique())
print("# of unique questions, raw questions:", uw_test_df['raw_question'].nunique())

(248836, 9)
# of rows with correct answer: 127153
# of unique questions: 2058
# of unique questions, raw questions: 2046


### compare questions in dataset

In [6]:
#compare datasets
qs_dbl_train = set(dbl_train_df.question)
qs_dbl_test = set(dbl_test_df.question)

qs_uw_train = set(uw_train_df.raw_question)
qs_uw_test = set(uw_test_df.raw_question)

In [ ]:
#compare within dataset -- check no train vs test overlap
overlap = qs_dbl_train.intersection(qs_dbl_test)
print(len(overlap))
print(overlap)


overlap = qs_uw_train.intersection(qs_uw_test) #-- seems like there is train/test overlap
print(len(overlap))
print(overlap)

0
set()
5
{'What prevents the stomach from being digested by itsown secretions?', 'Describe the various land biomes that are usually encounteredby a traveler going from the equator to the arcticpolar ice cap.', 'What is the difference between a kinesis and a taxis?', 'Which of the following is true?', ' Use the following key to translate the given formula of PL to natural, English sentences.'}


In [ ]:
#compare across datasets -- check overlap: train vs train and test vs test
overlap = qs_dbl_train.intersection(qs_uw_train)
print(len(overlap))
print(overlap)

overlap = qs_dbl_test.intersection(qs_uw_test)
print(len(overlap))
print(overlap)

5053
{"A nephew inherited a large parcel of unimproved land from his uncle. In need of cash, the nephew decided to sell the parcel. He contacted a real estate agent in the area about listing the parcel for sale. The nephew and the agent entered into a valid written contract whereby the agent promised to undertake best efforts to find a buyer for the parcel. The contract also provided that the agent was to be paid a commission of 5 percent on the gross sale price following the consummation of the sale and transfer of title. The agent succeeded in finding a buyer for the parcel. The agent notified the nephew that he had found a developer who wanted to purchase the parcel for $500,000. The agent handed the nephew a real estate sales contract, signed by the developer, in which the developer agreed to pay $500,000 for the purchase of the parcel. The nephew then signed the agreement himself. However, before consummation of the sale and transfer of title, the developer, without cause, repudia